# 18 · SHARP, AIA and fusion: earlier-data probability calibration

**Bamidele Akinwumi · 72-hour exploratory candidate-label experiment**

The SHARP and AIA three-seed models are already trained. This notebook fits only
probability calibration maps, on the reserved July–December 2014 cases. It can run
on the Mac once that complete role has been archived; it does not wait for later
GPU inference, retrain the neural networks, or score later outcomes.

## Context and methods

- Compare SHARP-only, AIA-only and the prespecified equal-probability average
  (`0.5 * SHARP + 0.5 * AIA`). Fusion weights are fixed, not selected here.
- Use identical valid-input cases for all three branches. Preserve every requested
  row, including failed inputs and their reasons, in the saved calibration table.
- Fit raw, Platt-style and isotonic candidates on July–November 2014, purging any
  outcome plus the assumed 24-hour reporting delay that extends into December.
- Select separately per branch on December 2014 by Brier score, then log loss,
  then declared method order. Refit all candidates on the complete earlier block.
- Reuse the SHARP training-fitted climatology and the tested notebook-02 methods.
  The selected map is saved before later evaluation; no future prediction archive
  enters this notebook.

## Assumptions and limits

These are candidate same-region M/X start labels. Continuous event coverage and
historical delivery are not established. Forecast windows are dependent; the
inner scores below describe method selection, not statistical significance or
independent validation. Conformal uncertainty, operational policy evaluation,
full-population availability and genuinely prospective validation remain separate.

## 1. Configuration and local inputs

Run using the repository training environment from the repository or `notebooks/`.
All source, cohort and parent-prediction hashes are checked. Every calibration
archive must be present and pass its original transfer checks. An incomplete role
stops execution rather than fitting on whichever blocks arrived first.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json, sys
import pandas as pd
from IPython.display import display, Markdown

REPO = Path.cwd().resolve()
if not (REPO / 'scripts/calibrate_multimodal72.py').is_file():
    REPO = REPO.parent
assert (REPO / 'scripts/calibrate_multimodal72.py').is_file(), 'Run inside the repository'
sys.path.insert(0, str(REPO))
__file__ = str(REPO / 'scripts/calibrate_multimodal72.py')
CONFIG = json.loads('{"calibration_start": "2014-07-01", "inner_selection_start": "2014-12-01", "calibration_end": "2015-01-01", "reporting_delay_hours": 24, "candidate_methods": ["raw", "platt", "isotonic"], "selection": "lowest inner December Brier, then log loss, then declared method order", "final_fit": "refit all prespecified candidates on the entire earlier probability_calibration block", "logit_clip_epsilon": 1e-06, "log_loss_clip_epsilon": 1e-12, "reliability_bins": 10, "experiment": "multimodal72_earlier_matched_calibration_v1", "created_utc": "2026-10-05T08:20:28.650974+00:00", "horizon_hours": 72, "target_scope": "primary", "inference_package": "outputs/aia72_inference_v3_20261004", "inference_contract_sha256": "95394d9f7db945c556e8600979e59837c26de6d28d910abbac11c725af3b4257", "case_manifest_sha256": "b687afd3d953f884b21c2bcab527a39de3a7ca127e0262b3f727b201d080a846", "sharp_training_dir": "outputs/sharp72_notebook_v1_20261002T080801721986Z", "sharp_summary_sha256": "e6418e5aac60a4ef7641ff32cd5c1b1b282db1e2b88af0a6f6aebc79a62058d0", "sharp_predictions_sha256": "7368c002a978321cdc67221564f3b7901687badc7aba28ab9df615179d1e017b", "fusion_weights": {"sharp": 0.5, "aia": 0.5}, "calibration_support": "Identical accepted SHARP/AIA cases for every branch; preserve failed-input rows", "evaluation_roles_read": [], "scientific_acceptance": false, "source_sha256": {"scripts/calibrate_multimodal72.py": "0559fdbe96bee6a1e77cb1e52286b7cb2aaf02633a464e87f8f9953bf5046497", "scripts/calibrate_sharp.py": "63b5d406447a73b402383caab1314fb98059c247adb4e3b7c34f7c20de7a33b1", "scripts/aia72_inference_archive_relay.py": "1131386270a4e1ed75eb5b6e6f2a767fe74d6ce1de19e1e4f9ef2c549d691c5b"}}')
OUTPUT = REPO / 'outputs' / ('multimodal72_calibration_v1_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
print('New result directory:', OUTPUT)
display(pd.DataFrame([
    {'Stage': 'Inner fitting', 'Period': 'July–November 2014; maturity purge at December boundary'},
    {'Stage': 'Method selection', 'Period': 'December 2014'},
    {'Stage': 'Final refit', 'Period': 'Complete eligible July–December 2014'},
    {'Stage': 'Later evaluation', 'Period': 'Not performed in this notebook'}
]))

New result directory: /Users/mac/Documents/ChatGPT/Imaging_Project/gray-box-solar-flare-forecasting/outputs/multimodal72_calibration_v1_20261005T082136654618Z


,Stage,Period
0,Inner fitting,July–November 2014; maturity purge at December...
1,Method selection,December 2014
2,Final refit,Complete eligible July–December 2014
3,Later evaluation,Not performed in this notebook


## 2. Executable methods

The code below is the exact tested calibration implementation. Its dependency hashes are pinned in the configuration.

In [2]:
"""Earlier-data calibration of frozen SHARP, AIA and equal-probability fusion.

This CPU-only stage can consume a completely archived calibration role while
inference continues on other roles. It never reads later prediction archives.
"""

from datetime import datetime, timezone
import gzip
import hashlib
import io
import json
from pathlib import Path
import tarfile

import numpy as np
import pandas as pd

from scripts.aia72_inference_archive_relay import verify_archive
from scripts.calibrate_sharp import (
    apply_calibrator, calibration_masks, check_probabilities, fit_calibrator,
    score_probabilities, sha256, save_json,
)

BRANCHES = ("sharp", "aia", "sharp_aia")


def require(value, message):
    if not value:
        raise ValueError(message)


def load_complete_role(repo, config):
    """Require every expected calibration block, including failed-input rows."""
    repo = Path(repo)
    package = repo / config["inference_package"]
    collection = package / "collected_blocks"
    contract_path = package / "bundle/configs/aia72_inference_v1.json"
    require(sha256(contract_path) == config["inference_contract_sha256"], "Inference contract changed")
    contract = json.loads(contract_path.read_text())
    require(contract["fitting_allowed"] is False and contract["seeds"] == [17, 29, 43], "Wrong inference experiment")
    sources = {"inference_contract_sha256": sha256(contract_path), "inputs": {}, "archives": {}}
    for name in ["cases.csv.gz", "objects.json", "blocks.json"]:
        digest = sha256(package / "inputs" / name)
        require(digest == contract["inputs"][name]["sha256"], "Inference input changed: " + name)
        sources["inputs"][name] = digest
    require(contract["inputs"]["cases.csv.gz"]["sha256"] == config["case_manifest_sha256"], "Case cohort changed")
    cases = pd.read_csv(package / "inputs/cases.csv.gz", float_precision="round_trip")
    require(not cases.forecast_case_id.duplicated().any(), "Duplicate manifest identities")
    role = "probability_calibration"
    expected = cases[cases.role.eq(role)].copy()
    blocks = [b for b in json.loads((package / "inputs/blocks.json").read_text()) if b["role"] == role]
    require(blocks and [v for b in blocks for v in b["case_ids"]] == expected.forecast_case_id.tolist(), "Block support/order differs")
    missing = [b["block_id"] for b in blocks if not (collection / (b["block_id"] + "_ack.json")).is_file()]
    require(not missing, "Calibration role incomplete: " + ", ".join(missing))
    pins = json.loads((package / "inputs/objects.json").read_text())
    # The archive verifier compares original CSV string values, including UTC text.
    import csv
    with gzip.open(package / "inputs/cases.csv.gz", "rt") as stream:
        case_rows = {r["forecast_case_id"]: r for r in csv.DictReader(stream)}
    rows = []
    for block in blocks:
        name = block["block_id"]
        ack_raw = (collection / (name + "_ack.json")).read_bytes()
        ack = json.loads(ack_raw)
        require(ack["status"] == "local_block_archive_verified", "Archive has no verified acknowledgement")
        raw = (collection / (name + ".tar.gz")).read_bytes()
        verified = verify_archive(raw, ack, block, pins, case_rows)
        require(all(ack[k] == v for k, v in verified.items()), "Acknowledgement does not match archive")
        with tarfile.open(fileobj=io.BytesIO(raw), mode="r:gz") as archive:
            payload = archive.extractfile(name + "_predictions.csv.gz").read()
        rows.append(pd.read_csv(io.BytesIO(payload), compression="gzip", float_precision="round_trip"))
        sources["archives"][name] = {"archive_sha256": verified["archive_sha256"],
                                      "ack_sha256": hashlib.sha256(ack_raw).hexdigest(),
                                      "cases": verified["cases"]}
    aia = pd.concat(rows, ignore_index=True)
    require(aia.forecast_case_id.tolist() == expected.forecast_case_id.tolist(), "Incomplete or reordered predictions")
    require(np.array_equal(aia.label, expected.label), "AIA labels changed")
    parent = repo / config["sharp_training_dir"]
    for name, key in [("summary.json", "sharp_summary_sha256"), ("predictions.csv.gz", "sharp_predictions_sha256")]:
        digest = sha256(parent / name)
        require(digest == config[key], "Frozen SHARP input changed: " + name)
        sources[name] = digest
    summary = json.loads((parent / "summary.json").read_text())
    require(summary["horizon"] == 72 and summary["scope"] == "primary", "Wrong SHARP target")
    sharp = pd.read_csv(parent / "predictions.csv.gz", float_precision="round_trip")
    require(not sharp.forecast_case_id.duplicated().any(), "Duplicate SHARP identities")
    sharp = sharp.set_index("forecast_case_id").reindex(expected.forecast_case_id)
    for name in ["role", "label", "issue_utc", "region_component_id"]:
        require(np.array_equal(sharp[name].to_numpy(), expected[name].to_numpy()), "SHARP case metadata changed: " + name)
    frame = expected.reset_index(drop=True)
    frame["sharp_raw"] = check_probabilities(sharp.probability_gru_mean.to_numpy(), frame.label.to_numpy())
    frame["aia_raw"] = aia.probability.to_numpy()
    frame["aia_input_status"] = aia.input_status.to_numpy()
    frame["aia_input_failure_reason"] = aia.input_failure_reason.fillna("").to_numpy()
    frame["sharp_aia_raw"] = .5 * frame.sharp_raw + .5 * frame.aia_raw
    sources["roles_read_from_AIA_archives"] = [role]
    sources["full_inference_completion_required_for_this_earlier_fit"] = False
    return frame, float(summary["train_climatology"]), sources


def fit_matched_calibration(frame, config, climatology):
    """Select on earlier December only; preserve every requested row separately."""
    require(frame.role.eq("probability_calibration").all(), "Only the reserved calibration role may enter this stage")
    require(not frame.forecast_case_id.duplicated().any(), "Duplicate calibration cases")
    require(frame.label_known.all() and frame.label.isin([0, 1]).all(), "Unknown outcomes in calibration cohort")
    require(frame.aia_input_status.isin(["ok", "missing_input", "invalid_input"]).all(), "Undeclared AIA input status")
    check_probabilities(frame.sharp_raw.to_numpy(), frame.label.to_numpy())
    available = frame.aia_input_status.eq("ok")
    check_probabilities(frame.loc[available, "aia_raw"].to_numpy())
    require(frame.loc[~available, "aia_raw"].isna().all(), "Failed input has an AIA probability")
    require(frame.loc[~available, "aia_input_failure_reason"].str.len().gt(0).all(), "Failed input needs a reason")
    expected_fusion = .5 * frame.sharp_raw + .5 * frame.aia_raw
    require(np.array_equal(frame.sharp_aia_raw, expected_fusion, equal_nan=True), "Fusion must use the fixed equal-probability mean")
    # Validate the time boundary on all requested cases before excluding failures.
    calibration_masks(frame, config)
    matched = frame.loc[available].copy()
    eligible, fit, select = calibration_masks(matched, config)
    choices, calibrators, scores = {}, {}, []
    for branch in BRANCHES:
        column = branch + "_raw"
        check_probabilities(matched[column].to_numpy(), matched.label.to_numpy())
        candidates = []
        for rank, method in enumerate(config["candidate_methods"]):
            parameters = fit_calibrator(matched.loc[fit, column].to_numpy(), matched.loc[fit, "label"].to_numpy(), method, config["logit_clip_epsilon"])
            p = apply_calibrator(matched.loc[select, column].to_numpy(), parameters)
            score = score_probabilities(matched.loc[select, "label"].to_numpy(), p, climatology, config, diagnostics=False)
            scores.append({"branch": branch, "method": method, **score})
            candidates.append((score["brier"], score["log_loss"], rank, method))
        choices[branch] = min(candidates)[-1]
        calibrators[branch] = {
            method: fit_calibrator(matched.loc[eligible, column].to_numpy(), matched.loc[eligible, "label"].to_numpy(), method, config["logit_clip_epsilon"])
            for method in config["candidate_methods"]
        }
    support = {name: {"cases": int(mask.sum()), "positive": int(matched.loc[mask, "label"].sum())}
               for name, mask in [("inner_fit", fit), ("inner_selection", select), ("final_fit", eligible),
                                  ("purged_at_inner_boundary", eligible & ~fit & ~select)]}
    return {"selected_methods": choices, "calibrators": calibrators, "inner_scores": scores,
            "matched_support": support, "requested_cases": len(frame), "input_failure_cases": int((~available).sum()),
            "fit_case_ids_sha256": hashlib.sha256("\n".join(matched.loc[eligible, "forecast_case_id"]).encode()).hexdigest(),
            "fusion_weights": {"sharp": .5, "aia": .5}, "train_climatology": climatology,
            "evaluation_roles_read": [], "scientific_acceptance": False}


def run(repo, config, output):
    """Persist source receipts and all requested calibration rows; no later scores."""
    repo, output = Path(repo), Path(output)
    require(not output.exists(), "Use a new output directory; preserve existing fits")
    for name, expected in config["source_sha256"].items():
        require(sha256(repo / name) == expected, "Calibration dependency changed: " + name)
    frame, climatology, sources = load_complete_role(repo, config)
    selection = fit_matched_calibration(frame, config, climatology)
    output.mkdir(parents=True)
    save_json(output / "config.json", config)
    save_json(output / "source_receipt.json", sources)
    save_json(output / "selection.json", selection)
    # Row preservation makes exclusions visible, rather than hiding them in fitting.
    columns = ["forecast_case_id", "role", "issue_utc", "outcome_end_utc_72h", "region_component_id", "label", "label_known",
               "sharp_raw", "aia_raw", "sharp_aia_raw", "aia_input_status", "aia_input_failure_reason"]
    frame[columns].to_csv(output / "calibration_cases.csv.gz", index=False, compression={"method": "gzip", "mtime": 0})
    pd.DataFrame(selection["inner_scores"]).to_csv(output / "inner_selection_scores.csv", index=False)
    summary = {"status": "completed_earlier_matched_calibration_not_evaluation",
               "completed_utc": datetime.now(timezone.utc).isoformat(),
               "selected_methods": selection["selected_methods"], "requested_cases": len(frame),
               "input_failure_cases": selection["input_failure_cases"], "matched_support": selection["matched_support"],
               "model_refits": 0, "GPU_calls": 0, "evaluation_roles_read": [],
               "scientific_acceptance": False, "source_sha256": sha256(Path(__file__)),
               "limitations": ["Candidate labels and historical source availability remain provisional.",
                               "Method choice on December 2014 does not establish later performance.",
                               "Full-population availability, conformal and operational policy evaluation remain separate stages.",
                               "Later Cycle-25/2026 results are retrospective; no independent or prospective validation is claimed."],
               "output_sha256": {p.name: sha256(p) for p in output.iterdir() if p.is_file()}}
    save_json(output / "summary.json", summary)
    return summary


## 3. Verify the complete earlier block, select and save the maps

In [3]:
summary = run(REPO, CONFIG, OUTPUT)
selection = json.loads((OUTPUT / 'selection.json').read_text())
print(summary['status'])
print('Later evaluation roles read:', summary['evaluation_roles_read'])
print('GPU calls:', summary['GPU_calls'])

completed_earlier_matched_calibration_not_evaluation
Later evaluation roles read: []
GPU calls: 0


## 4. Matched support and explicit exclusions

In [4]:
display(pd.DataFrame(summary['matched_support']).T)
display(pd.DataFrame([{'Requested calibration cases': summary['requested_cases'],
                       'Input failures preserved': summary['input_failure_cases']}]))

,cases,positive
inner_fit,1900,76
inner_selection,764,98
final_fit,2831,184
purged_at_inner_boundary,167,10


,Requested calibration cases,Input failures preserved
0,2831,0


## 5. Earlier method-selection evidence

These scores determine the choice. They do not establish which branch performs best in Cycle 25 or 2026.

In [5]:
scores = pd.DataFrame(selection['inner_scores'])
scores['selected'] = [method == selection['selected_methods'][branch]
                      for branch, method in zip(scores.branch, scores.method)]
display(scores[['branch', 'method', 'cases', 'positive', 'brier', 'log_loss', 'selected']].round(6))
display(pd.DataFrame([{'Branch': branch, 'Selected map': method}
                      for branch, method in selection['selected_methods'].items()]))

,branch,method,cases,positive,brier,log_loss,selected
0,sharp,raw,764,98,0.062285,0.216397,True
1,sharp,platt,764,98,0.087041,0.284485,False
2,sharp,isotonic,764,98,0.080173,0.609258,False
3,aia,raw,764,98,0.101253,0.319452,True
4,aia,platt,764,98,0.115175,0.424387,False
5,aia,isotonic,764,98,0.116370,0.538325,False
6,sharp_aia,raw,764,98,0.075362,0.231237,True
7,sharp_aia,platt,764,98,0.086189,0.278620,False
8,sharp_aia,isotonic,764,98,0.121418,0.594363,False


,Branch,Selected map
0,sharp,raw
1,aia,raw
2,sharp_aia,raw


## 6. Saved outputs and next stage

Use the frozen maps on the reserved uncertainty-calibration block, then evaluate the locked methods on matched later cases. No fusion improvement or operational readiness is inferred here.

In [6]:
display(Markdown('Saved calibrators: `' + str(OUTPUT / 'selection.json') + '`'))
print('Cases:', OUTPUT / 'calibration_cases.csv.gz')
print('Source receipts:', OUTPUT / 'source_receipt.json')
print('Status: earlier-data calibration complete; later evaluation remains pending.')

Saved calibrators: `/Users/mac/Documents/ChatGPT/Imaging_Project/gray-box-solar-flare-forecasting/outputs/multimodal72_calibration_v1_20261005T082136654618Z/selection.json`

Cases: /Users/mac/Documents/ChatGPT/Imaging_Project/gray-box-solar-flare-forecasting/outputs/multimodal72_calibration_v1_20261005T082136654618Z/calibration_cases.csv.gz
Source receipts: /Users/mac/Documents/ChatGPT/Imaging_Project/gray-box-solar-flare-forecasting/outputs/multimodal72_calibration_v1_20261005T082136654618Z/source_receipt.json
Status: earlier-data calibration complete; later evaluation remains pending.
